In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, roc_curve, auc

# Set aesthetic style for academic graphs
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

def generate_academic_visuals():
    print("Loading data and training model...")
    # Load your exact dataset
    df = pd.read_csv('../data/unizik_transactions.csv')
    
    feature_columns = [
        'device_student_count_24h', 'page_dwell_time_seconds', 'is_high_risk_asn',
        'failed_attempts_1h', 'session_hardware_mismatch', 'is_off_peak_hour'
    ]
    X = df[feature_columns]
    y = df['target_label']

    # Exact split from your train_model.py
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

    # Exact model constraints from your train_model.py
    model = DecisionTreeClassifier(criterion='gini', max_depth=5, min_samples_split=10, min_samples_leaf=5, random_state=42)
    model.fit(X_train, y_train)

    # ==========================================
    # 1. ACTUAL CONFUSION MATRIX HEATMAP
    # ==========================================
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    labels = ["Legitimate (0)", "Fraudulent (1)"]

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=labels, yticklabels=labels, annot_kws={"size": 14, "weight": "bold"})
    plt.title(f"Decision Tree Confusion Matrix (N = {len(y_test)})", fontsize=12, fontweight="bold", pad=12)
    plt.xlabel("Predicted Class", fontsize=11, fontweight="bold")
    plt.ylabel("Actual Class", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig("confusion_matrix_plot.png", dpi=300)
    plt.close()

    # ==========================================
    # 2. ACTUAL FEATURE IMPORTANCE BAR CHART
    # ==========================================
    display_names = [
        "Device Velocity (24h)", "Page Dwell Time", "High-Risk ASN",
        "Failed Attempts (1h)", "Hardware Mismatch", "Off-Peak Hour"
    ]
    importances = model.feature_importances_ * 100
    
    # Sort for plotting
    sorted_idx = np.argsort(importances)
    sorted_features = [display_names[i] for i in sorted_idx]
    sorted_importances = [importances[i] for i in sorted_idx]

    plt.figure(figsize=(8, 4.5))
    bars = plt.barh(sorted_features, sorted_importances, color="#1e3a8a", edgecolor="#0f172a", height=0.6)
    plt.title("Feature Importance by Gini Impurity Reduction (%)", fontsize=12, fontweight="bold", pad=12)
    plt.xlabel("Relative Importance (%)", fontsize=11, fontweight="bold")
    plt.xlim(0, max(sorted_importances) + 10)

    for bar in bars:
        width = bar.get_width()
        if width > 0:
            plt.text(width + 1.0, bar.get_y() + bar.get_height()/2, f"{width:.2f}%",
                     va="center", ha="left", fontsize=10, fontweight="bold", color="#1e293b")

    plt.tight_layout()
    plt.savefig("feature_importance_plot.png", dpi=300)
    plt.close()

    # ==========================================
    # 3. ACTUAL ROC CURVE
    # ==========================================
    # Get probabilities for the positive class (Fraudulent = 1)
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, color='#c62828', lw=2, label=f'Decision Tree (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='#64748b', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=11, fontweight="bold")
    plt.ylabel('True Positive Rate', fontsize=11, fontweight="bold")
    plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=12, fontweight="bold", pad=12)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig("roc_curve_plot.png", dpi=300)
    plt.close()

    print("Success! Saved: 'confusion_matrix_plot.png', 'feature_importance_plot.png', 'roc_curve_plot.png'.")
    from sklearn.tree import plot_tree

    # ==========================================
    # 4. ACTUAL DECISION TREE STRUCTURE PLOT
    # ==========================================
    print("Generating structural Decision Tree plot (this may take a moment)...")
    plt.figure(figsize=(24, 12))  # Large canvas to fit the branches
    
    plot_tree(
        model, 
        feature_names=feature_columns, 
        class_names=["Legitimate", "Fraudulent"], 
        filled=True, 
        rounded=True, 
        fontsize=9,
        proportion=True,
        precision=2
    )
    
    plt.title("Trained Decision Tree Logical Structure", fontsize=18, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.savefig("decision_tree_structure.png", dpi=300)
    plt.close()
    print("Success! Saved: 'decision_tree_structure.png'.")

if __name__ == '__main__':
    generate_academic_visuals()

Loading data and training model...
